# Ablations and native export parity: fundamentals

**Level:** 1  
**Prerequisites:** 05-evaluation-and-transfer  
**Reading time:** 15 minutes

## Objectives

The learning objective is to explain the problem and its data, then inspect and, at higher levels, execute the released method on its matched task. A complete evaluation must test whether each proposed component contributes and whether optimized model exports preserve the reference scorer. A matching score is numerical evidence only; it does not establish safer decisions, better task outcomes or generalization.

## Setup

The versions below are pinned in `notebooks/requirements.txt` and are also recorded in this notebook's metadata.

```bash
pip install reflexmesh==0.1.2 numpy==2.4.6 matplotlib==3.11.2 torch==2.14.1 joblib==1.6.0 gymnasium==1.4.0 scikit-learn==1.9.1 stable-baselines3==2.9.2 sb3-contrib==2.9.2
```

## Data

The notebook reads committed product artifacts. Owner: Neuraxis. Repository: https://github.com/fsantibanezleal/CAOS_Neuraxis. License: Apache-2.0. The JSON is a first-party, frozen experiment artifact; no external benchmark data are redistributed in this notebook.

This notebook reads `data/derived/index.json` and the listed trace artifacts. The method run uses an isolated Reflexmesh case workspace. For learned methods, fetch the exact public release assets with `python -m reflexmesh.cli fetch-models --destination models/checkpoints --version v0.01.000`, then set `NEURAXIS_CHECKPOINTS` to that folder. The release manifest verifies asset hashes. Planner methods M09/M10 are read from frozen traces unless `NEURAXIS_NOTEBOOK_MODEL` explicitly names a configured local Ollama model.

Artifacts for this lesson:

- `data/derived/index.json`
- `data/derived/research.json`
- `docs/evaluation/corrected-results.json`
- `data/derived/episodes/M03-C07-nominal-40000.json`
- `data/derived/episodes/M04-C07-nominal-40000.json`
- `data/derived/episodes/M11-C07-nominal-40000.json`
- `data/derived/episodes/M12-C07-nominal-40000.json`

## Method

Audit the canonical index digest against the research report, compare all four matched M12 ablations with the full controller, then inspect the native M03/M04 export vectors, threshold coverage, error tolerance and warmed latency scope.

For matched outcome \(Y\), the ablation contrast is \(\Delta=Y_{full}-Y_{ablated}\); paired seeds and fixed task templates determine the unit of uncertainty. Export parity is \(\max_i |p_i^{native}-p_i^{reference}|\le\epsilon\), a separate numerical criterion.

### Check the pinned analysis environment.

In [1]:
# Setup is performed outside the notebook; report the exact runtime pins.
import importlib.metadata
for distribution in ('reflexmesh', 'numpy', 'matplotlib', 'torch', 'joblib', 'gymnasium', 'scikit-learn', 'stable-baselines3', 'sb3-contrib'):
    print('{}=={}'.format(distribution, importlib.metadata.version(distribution)))

reflexmesh==0.1.2
numpy==2.4.6
matplotlib==3.11.2
torch==2.14.1
joblib==1.6.0
gymnasium==1.4.0
scikit-learn==1.9.1
stable-baselines3==2.9.2
sb3-contrib==2.9.2


### Verify the audit and each trace's bytes before interpreting outcomes.

In [2]:
# Read committed evidence and use the released API; never define another policy.
import hashlib, json
from pathlib import Path
ROOT = Path('../../../').resolve()
index_path = ROOT / 'data/derived/index.json'
index_bytes = index_path.read_bytes()
index = json.loads(index_bytes)
audit = json.loads((ROOT / 'data/derived/benchmark-adequacy.json').read_text(encoding='utf-8'))
assert hashlib.sha256(index_bytes).hexdigest() == audit['index_sha256']
selected_methods = ['M03', 'M04', 'M11', 'M12']
sample_case = 'C07'
seed = 40000
records = {}
for method_id in selected_methods:
    row = next(r for r in index['episodes'] if r['method_id'] == method_id and r['case_id'] == sample_case and r['variant'] == 'nominal' and r['seed'] == seed)
    trace_path = ROOT / 'data/derived' / row['path']
    trace_bytes = trace_path.read_bytes()
    assert hashlib.sha256(trace_bytes).hexdigest() == row['sha256']
    records[method_id] = json.loads(trace_bytes)
print('Index digest:', hashlib.sha256(index_bytes).hexdigest())
print('Evidence records:', len(records), '| case:', sample_case, '| seed:', seed)
print('Methods:', ', '.join(selected_methods))

Index digest: 1024a451fdc1f1aebc4e8d8d66e6bf1aca119f26ae058d3de9b84ff312d8d7a6
Evidence records: 4 | case: C07 | seed: 40000
Methods: M03, M04, M11, M12


### Inspect the original decisions and outcomes

In [3]:
# Summarize the recorded first action, authorized candidates, and independent oracle outcome.
for method_id, episode in records.items():
    first = episode['decisions'][0]
    allowed = [candidate['action'] for candidate in first['candidates'] if candidate['allowed']]
    print(method_id, '| action=', first['action'], '| reason=', first['reason'], '| allowed=', allowed)
    print('  success=', episode['metrics']['task_success'], '| steps=', episode['metrics']['steps'], '| violations=', episode['metrics']['violations'])

M03 | action= a-cancel | reason= calibrated_demonstration_preference | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M04 | action= a-cancel | reason= calibrated_demonstration_preference | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M11 | action= a-cancel | reason= learned_incremental_benefit_gate | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M12 | action= a-cancel | reason= urgent_cancellation_during_deliberation | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0


## Results

All four ablations preserve the same 1,160/1,200 task successes, so this experiment does not show success contribution by memory, effects, deferral or invalidation. Runtime budgets differ across variants, and measured export timings exclude Python feature construction. Numerical parity is not policy safety calibration.

No generated outputs are committed by this authoring step; `scripts/run_notebooks.py` executes and stores them from a fresh kernel.

## Summary

The negative ablation result is material: the proposed components do not change success on this saturated suite. Native exports meet their recorded numerical tolerance, which supports implementation consistency only.

## References

- [XGB](https://arxiv.org/abs/1603.02754)
- [PPO](https://arxiv.org/abs/1707.06347)